<a href="https://colab.research.google.com/github/Prathamesh-exe/AI-ML-Colabs-and-Gists/blob/main/rlmsmol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip -q install openai
from openai import OpenAI
from pathlib import Path
from collections import Counter
import re
import json
import time
import uuid

In [28]:
from openai import OpenAI
import uuid

OPENCODE_API_KEY = "oc_sk_NICETRY"

SESSION_ID = str(uuid.uuid4())

client = OpenAI(
    api_key=OPENCODE_API_KEY,
    base_url="https://opencode.ai/zen/go/v1",
    default_headers={
        "x-opencode-session": SESSION_ID,
        "User-Agent": "rlm-log-detective/1.0"
    }
)

MODEL = "deepseek-v4-flash"

print("✓ OpenCode client initialized")
print(f"✓ Model: {MODEL}")
print(f"✓ Session: {SESSION_ID}")

✓ OpenCode client initialized
✓ Model: deepseek-v4-flash
✓ Session: 7a614c7e-e8ab-4d4e-8b30-d6ef0da5b686


In [10]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain Recursive Language Models in 1 sentence."
        }
    ]
)

print(response.choices[0].message.content)

Recursive language models generate or score sequences by repeatedly applying the same function to previously produced representations, enabling them to handle arbitrarily long sequences with fixed parameters.


In [29]:
LOG_FILE = Path("/content/OpenStack_2k.log")

if not LOG_FILE.exists():
    raise FileNotFoundError(
        f"Log file not found: {LOG_FILE}"
    )

start = time.perf_counter()

with LOG_FILE.open("r", errors="ignore") as f:
    RAW_LOGS = f.readlines()

elapsed = time.perf_counter() - start

print("\n" + "═" * 70)
print("  DATASET")
print("═" * 70)

print(f"File       : {LOG_FILE}")
print(f"Size       : {LOG_FILE.stat().st_size / 1024 / 1024:.2f} MB")
print(f"Lines      : {len(RAW_LOGS):,}")
print(f"Read time  : {elapsed:.3f}s")

print("\nFirst 5 log lines:\n")

for line in RAW_LOGS[:5]:
    print(line.rstrip())


══════════════════════════════════════════════════════════════════════
  DATASET
══════════════════════════════════════════════════════════════════════
File       : /content/OpenStack_2k.log
Size       : 0.57 MB
Lines      : 2,000
Read time  : 0.002s

First 5 log lines:

nova-api.log.1.2017-05-16_13:53:08 2017-05-16 00:00:00.008 25746 INFO nova.osapi_compute.wsgi.server [req-38101a0b-2096-447d-96ea-a692162415ae 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2477829
nova-api.log.1.2017-05-16_13:53:08 2017-05-16 00:00:00.272 25746 INFO nova.osapi_compute.wsgi.server [req-9bc36dd9-91c5-4314-898a-47625eb93b09 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2577181
nova-api.log.1.2017-05-16_13:53:08 2017-05-16 00:00:01.551 2574

In [30]:
LOG_PATTERN = re.compile(
    r"""
    ^(?P<source>\S+)
    \s+
    (?P<timestamp>\d{4}-\d{2}-\d{2}
                     \s+
                     \d{2}:\d{2}:\d{2}\.\d+)
    \s+
    (?P<pid>\d+)
    \s+
    (?P<level>DEBUG|INFO|WARNING|WARN|ERROR|CRITICAL)
    \s+
    (?P<service>\S+)
    \s+
    (?P<message>.*)
    $
    """,
    re.VERBOSE
)


def parse_log_line(line, line_number):

    raw = line.strip()

    match = LOG_PATTERN.match(raw)

    if not match:
        return {
            "line_number": line_number,
            "raw": raw,
            "parse_error": True
        }

    data = match.groupdict()

    data["line_number"] = line_number
    data["raw"] = raw
    data["parse_error"] = False

    return data


start = time.perf_counter()

PARSED_LOGS = [
    parse_log_line(line, i)
    for i, line in enumerate(RAW_LOGS, start=1)
]

elapsed = time.perf_counter() - start

parse_errors = sum(
    x["parse_error"]
    for x in PARSED_LOGS
)

print("\n" + "═" * 70)
print("  LOG PARSER")
print("═" * 70)

print(f"Parsed records : {len(PARSED_LOGS):,}")
print(f"Parse errors   : {parse_errors}")
print(f"Parse time     : {elapsed:.3f}s")


══════════════════════════════════════════════════════════════════════
  LOG PARSER
══════════════════════════════════════════════════════════════════════
Parsed records : 2,000
Parse errors   : 0
Parse time     : 0.006s


In [32]:
def dataset_summary():

    valid = [
        x for x in PARSED_LOGS
        if not x["parse_error"]
    ]

    levels = Counter(
        x["level"]
        for x in valid
    )

    services = Counter(
        x["service"]
        for x in valid
    )

    return {
        "total_lines": len(PARSED_LOGS),
        "parsed_lines": len(valid),
        "parse_errors": len(PARSED_LOGS) - len(valid),
        "log_levels": dict(levels),
        "top_services": dict(
            services.most_common(10)
        )
    }


summary = dataset_summary()

print("\n[DATASET SUMMARY]")
print(
    json.dumps(
        summary,
        indent=2
    )
)


[DATASET SUMMARY]
{
  "total_lines": 2000,
  "parsed_lines": 2000,
  "parse_errors": 0,
  "log_levels": {
    "INFO": 1969,
    "WARNING": 31
  },
  "top_services": {
    "nova.osapi_compute.wsgi.server": 809,
    "nova.virt.libvirt.imagecache": 336,
    "nova.compute.manager": 262,
    "nova.metadata.wsgi.server": 208,
    "nova.compute.claims": 168,
    "nova.virt.libvirt.driver": 107,
    "nova.compute.resource_tracker": 60,
    "nova.api.openstack.compute.server_external_events": 22,
    "nova.api.openstack.wsgi": 21,
    "nova.scheduler.host_manager": 7
  }
}


In [31]:
def warning_summary(max_results=50):

    warnings = [
        x for x in PARSED_LOGS
        if x.get("level") == "WARNING"
    ]

    services = Counter(
        x["service"]
        for x in warnings
    )

    messages = Counter(
        x["message"]
        for x in warnings
    )

    return {
        "total_warnings": len(warnings),
        "services": dict(services),
        "message_patterns": [
            {
                "message": message,
                "count": count
            }
            for message, count
            in messages.most_common(max_results)
        ]
    }


def search_messages(
    query,
    max_results=20
):

    query = query.lower()

    matches = [
        x
        for x in PARSED_LOGS
        if query in x.get(
            "message",
            ""
        ).lower()
    ]

    return {
        "query": query,
        "total_matches": len(matches),
        "results": matches[:max_results]
    }


def group_messages(
    level=None,
    service=None,
    top_n=20
):

    records = PARSED_LOGS

    if level:
        records = [
            x
            for x in records
            if x.get("level") == level.upper()
        ]

    if service:
        records = [
            x
            for x in records
            if service.lower()
            in x.get(
                "service",
                ""
            ).lower()
        ]

    groups = Counter(
        x.get(
            "message",
            ""
        )
        for x in records
    )

    return {
        "filters": {
            "level": level,
            "service": service
        },
        "total_records": len(records),
        "groups": [
            {
                "message": message,
                "count": count
            }
            for message, count
            in groups.most_common(top_n)
        ]
    }


def service_summary(
    service,
    top_n=20
):

    records = [
        x
        for x in PARSED_LOGS
        if service.lower()
        in x.get(
            "service",
            ""
        ).lower()
    ]

    levels = Counter(
        x["level"]
        for x in records
    )

    messages = Counter(
        x["message"]
        for x in records
    )

    return {
        "service": service,
        "total_records": len(records),
        "log_levels": dict(levels),
        "top_messages": [
            {
                "message": message,
                "count": count
            }
            for message, count
            in messages.most_common(top_n)
        ]
    }


def get_context(
    line_number,
    radius=3
):

    start = max(
        1,
        line_number - radius
    )

    end = min(
        len(PARSED_LOGS),
        line_number + radius
    )

    records = PARSED_LOGS[
        start - 1:end
    ]

    return {
        "target_line": line_number,
        "range": {
            "start": start,
            "end": end
        },
        "results": records
    }

✓ RLM engine initialized.
✓ DeepSeek can recursively investigate the logs.


In [33]:
# STEP 5 — BETTER LOG ANALYSIS TOOLS

import re
import json
import time
from collections import Counter


# Rebuild parsed records with the original log line preserved

def parse_log_line(line, line_number):
    match = LOG_PATTERN.match(line.strip())

    if not match:
        return {
            "line_number": line_number,
            "raw": line.strip(),
            "parse_error": True
        }

    data = match.groupdict()
    data["line_number"] = line_number
    data["raw"] = line.strip()
    data["parse_error"] = False

    return data


PARSED_LOGS = [
    parse_log_line(line, i)
    for i, line in enumerate(RAW_LOGS, start=1)
]


# Dataset overview

def dataset_summary():
    valid = [
        x for x in PARSED_LOGS
        if not x["parse_error"]
    ]

    levels = Counter(x["level"] for x in valid)
    services = Counter(x["service"] for x in valid)

    return {
        "total_lines": len(PARSED_LOGS),
        "parsed_lines": len(valid),
        "parse_errors": len(PARSED_LOGS) - len(valid),
        "log_levels": dict(levels),
        "top_services": dict(services.most_common(10))
    }


# Warning overview

def warning_summary(max_results=50):
    warnings = [
        x for x in PARSED_LOGS
        if x.get("level") == "WARNING"
    ]

    services = Counter(
        x["service"]
        for x in warnings
    )

    messages = Counter(
        x["message"]
        for x in warnings
    )

    return {
        "total_warnings": len(warnings),
        "services": dict(services),
        "message_patterns": [
            {
                "message": message,
                "count": count
            }
            for message, count
            in messages.most_common(max_results)
        ]
    }


# Search message content

def search_messages(
    query,
    max_results=20
):
    query = query.lower()

    matches = [
        x for x in PARSED_LOGS
        if query in x.get("message", "").lower()
    ]

    return {
        "query": query,
        "total_matches": len(matches),
        "results": matches[:max_results]
    }


# Group similar messages

def group_messages(
    level=None,
    service=None,
    top_n=20
):
    records = PARSED_LOGS

    if level:
        records = [
            x for x in records
            if x.get("level") == level.upper()
        ]

    if service:
        records = [
            x for x in records
            if service.lower()
            in x.get("service", "").lower()
        ]

    groups = Counter(
        x.get("message", "")
        for x in records
    )

    return {
        "filters": {
            "level": level,
            "service": service
        },
        "total_records": len(records),
        "groups": [
            {
                "message": message,
                "count": count
            }
            for message, count
            in groups.most_common(top_n)
        ]
    }


# Service statistics

def service_summary(
    service,
    top_n=20
):
    records = [
        x for x in PARSED_LOGS
        if service.lower()
        in x.get("service", "").lower()
    ]

    levels = Counter(
        x["level"]
        for x in records
    )

    messages = Counter(
        x["message"]
        for x in records
    )

    return {
        "service": service,
        "total_records": len(records),
        "log_levels": dict(levels),
        "top_messages": [
            {
                "message": message,
                "count": count
            }
            for message, count
            in messages.most_common(top_n)
        ]
    }


# Retrieve records around a specific line

def get_context(
    line_number,
    radius=3
):
    start = max(1, line_number - radius)
    end = min(len(PARSED_LOGS), line_number + radius)

    records = PARSED_LOGS[start - 1:end]

    return {
        "target_line": line_number,
        "range": {
            "start": start,
            "end": end
        },
        "results": records
    }


# Register the investigation tools

TOOLS = {
    "dataset_summary": dataset_summary,
    "warning_summary": warning_summary,
    "search_messages": search_messages,
    "group_messages": group_messages,
    "service_summary": service_summary,
    "get_context": get_context,
}


print("\n" + "═" * 70)
print("  INVESTIGATION TOOLSET")
print("═" * 70)

for name in TOOLS:
    print(f"  • {name}")

print(f"\n✓ {len(TOOLS)} investigation tools registered.")


# Test the important path

print("\n[WARNING ANALYSIS]")

result = warning_summary()

print(
    json.dumps(
        result,
        indent=2,
        default=str
    )
)


══════════════════════════════════════════════════════════════════════
  INVESTIGATION TOOLSET
══════════════════════════════════════════════════════════════════════
  • dataset_summary
  • warning_summary
  • search_messages
  • group_messages
  • service_summary
  • get_context

✓ 6 investigation tools registered.

[WARNING ANALYSIS]
{
  "total_warnings": 31,
  "services": {
    "nova.virt.libvirt.imagecache": 30,
    "nova.compute.manager": 1
  },
  "message_patterns": [
    {
      "message": "[req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] Unknown base file: /var/lib/nova/instances/_base/a489c868f0c37da93b76227c91bb03908ac0e742",
      "count": 30
    },
    {
      "message": "[req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] While synchronizing instance power states, found 1 instances in the database and 0 instances on the hypervisor.",
      "count": 1
    }
  ]
}


In [34]:
TOOLS = {
    "dataset_summary": dataset_summary,
    "warning_summary": warning_summary,
    "search_messages": search_messages,
    "group_messages": group_messages,
    "service_summary": service_summary,
    "get_context": get_context,
}


print("\n" + "═" * 70)
print("  INVESTIGATION TOOLSET")
print("═" * 70)

for name in TOOLS:
    print(f"  • {name}")

print(f"\n✓ {len(TOOLS)} investigation tools registered.")


══════════════════════════════════════════════════════════════════════
  INVESTIGATION TOOLSET
══════════════════════════════════════════════════════════════════════
  • dataset_summary
  • warning_summary
  • search_messages
  • group_messages
  • service_summary
  • get_context

✓ 6 investigation tools registered.


In [35]:
print("\n[WARNING ANALYSIS]")

result = warning_summary()

print(
    json.dumps(
        result,
        indent=2,
        default=str
    )
)


[WARNING ANALYSIS]
{
  "total_warnings": 31,
  "services": {
    "nova.virt.libvirt.imagecache": 30,
    "nova.compute.manager": 1
  },
  "message_patterns": [
    {
      "message": "[req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] Unknown base file: /var/lib/nova/instances/_base/a489c868f0c37da93b76227c91bb03908ac0e742",
      "count": 30
    },
    {
      "message": "[req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] While synchronizing instance power states, found 1 instances in the database and 0 instances on the hypervisor.",
      "count": 1
    }
  ]
}


In [36]:
RLM_SYSTEM_PROMPT = """
You are an autonomous Recursive Language Model investigating an OpenStack
system-log dataset.

The complete dataset exists outside your context inside a Python analysis
environment.

You do NOT have direct access to the dataset.

You can only learn about the dataset through the investigation tools.

Your job is to investigate the user's question systematically and produce
an evidence-based conclusion.

CORE RULES

1. Never claim to have inspected the complete dataset.

2. Never invent log entries, counts, events, causes, relationships, or
   timestamps.

3. Every factual claim about the dataset must be supported by information
   returned by a tool.

4. Your general OpenStack knowledge may be used to interpret observed
   evidence, but must never be used as a substitute for evidence.

5. Do not perform blind regex experimentation.

6. Do not repeatedly search for increasingly generic patterns such as:
   ".*", ".", "^", "$", numbers, single characters, or common words.

7. Prefer the semantic investigation tools over low-level searching.

8. Investigate progressively. Each tool call should answer a specific
   question raised by the previous evidence.

INVESTIGATION STATE

Maintain these five pieces of state mentally throughout the investigation:

KNOWN FACTS
Facts directly established by tool results.

HYPOTHESES
Possible explanations that are not yet proven.

EVIDENCE NEEDED
What evidence would distinguish between the important hypotheses.

NEXT ACTION
The single most useful next tool call.

After every important tool result, update this state before deciding
what to do next.

INVESTIGATION STRATEGY

Begin with a broad structural understanding of the dataset.

Use:

dataset_summary

to understand the total dataset, log levels, and major services.

If the investigation concerns warnings, use:

warning_summary

to determine:

- total warning count
- services producing warnings
- repeated warning patterns
- frequency of each warning pattern

Do not retrieve every warning simply because they exist.

Instead, identify the most significant pattern first.

Use:

group_messages

when you need to determine whether messages form repeated patterns
within a particular log level or service.

Use:

service_summary

when a particular OpenStack service becomes relevant and you need
to understand what that service is doing in the dataset.

Use:

search_messages

only for a targeted investigation after you have identified a concrete
keyword, message pattern, component, identifier, or operation that needs
verification.

Use:

get_context

when you have a specific line number and need to inspect nearby events
to determine temporal or operational relationships.

HYPOTHESIS-DRIVEN INVESTIGATION

Do not stop after discovering a repeated message.

For example, if a warning appears many times:

1. Establish how frequently it occurs.
2. Identify which service produces it.
3. Identify the exact repeated message pattern.
4. Determine whether the repeated events reference the same request,
   instance, resource, path, or component.
5. Inspect relevant surrounding events.
6. Search for related successful, informational, or additional warning
   messages.
7. Determine whether the evidence supports a common underlying operation.
8. Only then form a conclusion.

IMPORTANT DISTINCTION

Always distinguish between:

OBSERVATION
Something directly returned by a tool.

HYPOTHESIS
A possible explanation inferred from observations.

CONCLUSION
An explanation supported by sufficient evidence.

Do not present a hypothesis as a confirmed cause.

TOOL SELECTION

Choose the tool based on the question you need to answer.

Use dataset_summary for:
- overall dataset structure
- log-level distribution
- service distribution

Use warning_summary for:
- warning frequency
- repeated warning patterns
- warning-producing services

Use group_messages for:
- repeated message patterns
- message frequency within a level or service

Use search_messages for:
- targeted keyword/message investigation

Use service_summary for:
- deeper investigation of a specific service

Use get_context for:
- events immediately surrounding a known log line

Do not call a tool unless you can explain why its result is useful
for the current investigation.

RECURSIVE BEHAVIOR

The investigation should be recursive in the following sense:

A tool result changes what you know.

What you know determines the next question.

The next question determines the next tool call.

Continue this process until the important hypotheses have enough evidence
to distinguish between them.

Do not use all available tools merely for the sake of using them.

STOPPING CONDITIONS

Stop investigating when:

- the main recurring pattern has been identified,
- its frequency and source are established,
- relevant relationships have been investigated,
- the strongest explanation is supported by the available evidence,
- and additional tool calls are unlikely to materially change the conclusion.

If the evidence is insufficient, explicitly say what cannot be established.

FINAL RESPONSE

Your final answer must contain:

Finding
A concise description of the main observed behavior.

Evidence
The important counts, services, messages, identifiers, and related events
returned by the tools.

Investigation
Briefly explain how the evidence was progressively investigated.

Conclusion
State the most plausible explanation supported by the dataset.

Confidence
Use:
- High
- Medium
- Low

Base confidence on the amount and consistency of evidence, not intuition.

Limitations
Explain what the dataset or available tools cannot establish.

Do not fabricate missing evidence.

Remember:

The dataset is external computational memory.

Your context contains reasoning and compact evidence.

The Python tools contain the actual dataset.

Investigate first.
Reason from evidence.
Then conclude.
"""

In [37]:
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "dataset_summary",
            "description": (
                "Get a high-level overview of the OpenStack log dataset. "
                "Use this first to understand total records, log levels, "
                "and major services."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "warning_summary",
            "description": (
                "Analyze WARNING records as groups. Returns the total "
                "warning count, services producing warnings, and repeated "
                "warning message patterns with frequencies."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "max_results": {
                        "type": "integer",
                        "description": (
                            "Maximum number of warning patterns to return."
                        ),
                        "default": 20
                    }
                },
                "required": []
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "search_messages",
            "description": (
                "Search log message content for a specific keyword or phrase. "
                "Use this for targeted investigation after identifying "
                "something concrete that needs verification."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "Specific keyword, phrase, identifier, "
                            "resource name, or component to search for."
                        )
                    },
                    "max_results": {
                        "type": "integer",
                        "description": (
                            "Maximum matching records to return."
                        ),
                        "default": 20
                    }
                },
                "required": ["query"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "group_messages",
            "description": (
                "Group log messages by exact message text and return "
                "the most frequent patterns. Can optionally filter "
                "by log level or service."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "level": {
                        "type": "string",
                        "description": (
                            "Optional log level such as INFO, WARNING, "
                            "ERROR, or CRITICAL."
                        )
                    },
                    "service": {
                        "type": "string",
                        "description": (
                            "Optional OpenStack service name or partial "
                            "service name."
                        )
                    },
                    "top_n": {
                        "type": "integer",
                        "description": (
                            "Number of message groups to return."
                        ),
                        "default": 20
                    }
                },
                "required": []
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "service_summary",
            "description": (
                "Analyze a specific OpenStack service. Returns its total "
                "records, log-level distribution, and most frequent messages."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "service": {
                        "type": "string",
                        "description": (
                            "OpenStack service name or partial service name."
                        )
                    },
                    "top_n": {
                        "type": "integer",
                        "description": (
                            "Number of top messages to return."
                        ),
                        "default": 20
                    }
                },
                "required": ["service"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_context",
            "description": (
                "Retrieve log records immediately before and after a "
                "specific line number. Use this to investigate temporal "
                "relationships around an identified event."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "line_number": {
                        "type": "integer",
                        "description": (
                            "The dataset line number to investigate."
                        )
                    },
                    "radius": {
                        "type": "integer",
                        "description": (
                            "Number of lines before and after the target "
                            "line to retrieve."
                        ),
                        "default": 3
                    }
                },
                "required": ["line_number"]
            }
        }
    }
]


print("\n" + "═" * 70)
print("  RLM TOOL DEFINITIONS")
print("═" * 70)

for tool in TOOL_DEFINITIONS:
    print(
        f"  • {tool['function']['name']}"
    )

print(
    f"\n✓ {len(TOOL_DEFINITIONS)} tools exposed to DeepSeek."
)


══════════════════════════════════════════════════════════════════════
  RLM TOOL DEFINITIONS
══════════════════════════════════════════════════════════════════════
  • dataset_summary
  • warning_summary
  • search_messages
  • group_messages
  • service_summary
  • get_context

✓ 6 tools exposed to DeepSeek.


In [38]:
def print_tool_preview(
    tool_name,
    result
):

    print("\n   PYTHON RESULT")

    if tool_name == "dataset_summary":

        print(
            f"   • Lines        : "
            f"{result['total_lines']}"
        )

        print(
            f"   • Parsed       : "
            f"{result['parsed_lines']}"
        )

        print(
            f"   • Parse errors : "
            f"{result['parse_errors']}"
        )

        print(
            f"   • Levels       : "
            f"{result['log_levels']}"
        )

    elif tool_name == "warning_summary":

        print(
            f"   • Total warnings : "
            f"{result['total_warnings']}"
        )

        print(
            f"   • Services       : "
            f"{result['services']}"
        )

        print("   • Patterns:")

        for pattern in result["message_patterns"][:5]:

            message = pattern["message"]

            if len(message) > 100:
                message = message[:100] + "..."

            print(
                f"       {pattern['count']}x  "
                f"{message}"
            )

    elif tool_name == "search_messages":

        print(
            f"   • Query         : "
            f"{result['query']}"
        )

        print(
            f"   • Total matches : "
            f"{result['total_matches']}"
        )

        for record in result["results"][:3]:

            print(
                f"       line {record['line_number']} | "
                f"{record['level']} | "
                f"{record['service']}"
            )

            message = record["message"]

            if len(message) > 120:
                message = message[:120] + "..."

            print(
                f"       {message}"
            )

    elif tool_name == "group_messages":

        print(
            f"   • Records : "
            f"{result['total_records']}"
        )

        for group in result["groups"][:5]:

            message = group["message"]

            if len(message) > 100:
                message = message[:100] + "..."

            print(
                f"       {group['count']}x  "
                f"{message}"
            )

    elif tool_name == "service_summary":

        print(
            f"   • Service : "
            f"{result['service']}"
        )

        print(
            f"   • Records : "
            f"{result['total_records']}"
        )

        print(
            f"   • Levels  : "
            f"{result['log_levels']}"
        )

        print("   • Top messages:")

        for item in result["top_messages"][:5]:

            message = item["message"]

            if len(message) > 100:
                message = message[:100] + "..."

            print(
                f"       {item['count']}x  "
                f"{message}"
            )

    elif tool_name == "get_context":

        print(
            f"   • Lines "
            f"{result['range']['start']}"
            f"-{result['range']['end']}"
        )

        for record in result["results"]:

            message = record.get(
                "message",
                ""
            )

            if len(message) > 110:
                message = message[:110] + "..."

            print(
                f"       "
                f"{record['line_number']:>5} | "
                f"{record.get('level', '?'):<8} | "
                f"{message}"
            )

    else:

        preview = json.dumps(
            result,
            indent=2,
            default=str
        )

        print(
            preview[:1500]
        )

In [39]:
def run_rlm(
    question,
    max_steps=10
):

    messages = [
        {
            "role": "system",
            "content": RLM_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        }
    ]

    print("\n" + "═" * 75)
    print("  RLM INVESTIGATION STARTED")
    print("═" * 75)

    print(
        f"\nQuestion:\n"
        f"{question.strip()}"
    )

    print(
        f"\nMaximum steps: "
        f"{max_steps}"
    )

    for step in range(
        1,
        max_steps + 1
    ):

        print("\n" + "─" * 75)
        print(
            f"  RLM STEP "
            f"{step}/{max_steps}"
        )
        print("─" * 75)

        start = time.perf_counter()

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto"
        )

        elapsed = (
            time.perf_counter()
            - start
        )

        message = response.choices[0].message

        print(
            f"Model response time: "
            f"{elapsed:.2f}s"
        )

        assistant_data = {
            "role": "assistant",
            "content": message.content
        }

        if message.tool_calls:

            assistant_data["tool_calls"] = [
                {
                    "id": call.id,
                    "type": "function",
                    "function": {
                        "name": call.function.name,
                        "arguments": call.function.arguments
                    }
                }
                for call in message.tool_calls
            ]

        messages.append(
            assistant_data
        )

        if message.content:

            print("\n💬 MODEL")
            print(
                message.content
            )

        if not message.tool_calls:

            print("\n" + "═" * 75)
            print(
                "  RLM INVESTIGATION COMPLETE"
            )
            print("═" * 75)

            return message.content

        print(
            f"\n🔧 "
            f"{len(message.tool_calls)} "
            f"tool call(s) requested"
        )

        for call in message.tool_calls:

            tool_name = call.function.name

            try:
                arguments = json.loads(
                    call.function.arguments
                )
            except Exception:
                arguments = {}

            print("\n🔧 TOOL CALL")

            print(
                f"   Name      : "
                f"{tool_name}"
            )

            print(
                f"   Arguments : "
                f"{json.dumps(arguments, indent=2)}"
            )

            if tool_name not in TOOLS:

                result = {
                    "error":
                    f"Unknown tool: {tool_name}"
                }

                print(
                    "   ERROR     : "
                    "Unknown tool"
                )

            else:

                try:

                    tool_start = (
                        time.perf_counter()
                    )

                    result = TOOLS[
                        tool_name
                    ](
                        **arguments
                    )

                    tool_elapsed = (
                        time.perf_counter()
                        - tool_start
                    )

                    result_text = json.dumps(
                        result,
                        default=str
                    )

                    print(
                        f"   Time      : "
                        f"{tool_elapsed:.3f}s"
                    )

                    print(
                        f"   Result size: "
                        f"{len(result_text):,} chars"
                    )

                    print_tool_preview(
                        tool_name,
                        result
                    )

                except Exception as e:

                    result = {
                        "error": str(e)
                    }

                    print(
                        f"   ERROR     : "
                        f"{e}"
                    )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(
                        result,
                        default=str
                    )
                }
            )

        print(
            "\n↻ Tool results returned "
            "to DeepSeek."
        )

    print("\n" + "═" * 75)
    print(
        "  MAXIMUM STEPS REACHED"
    )
    print("═" * 75)

    return None

In [40]:
question = """
Investigate the WARNING events in this OpenStack log dataset.

Determine:

1. What distinct warning types are occurring?
2. Which OpenStack services generate them?
3. Which warning pattern is most frequent?
4. Are repeated warnings related to the same resource or operation?
5. Is there evidence in related or nearby log messages that explains
   the recurring warning?
6. What is the most plausible explanation supported by the dataset?

Investigate progressively.

Do not simply retrieve all WARNING records and summarize them.

Start with the warning population, identify important patterns,
quantify them, and drill into the most significant pattern.

Use additional tool calls to verify important hypotheses before
reaching a conclusion.

Clearly distinguish observations from hypotheses and conclusions.

If the available logs are insufficient to establish the actual cause,
clearly state what is proven and what remains uncertain.
"""

result = run_rlm(
    question,
    max_steps=10
)


═══════════════════════════════════════════════════════════════════════════
  RLM INVESTIGATION STARTED
═══════════════════════════════════════════════════════════════════════════

Question:
Investigate the WARNING events in this OpenStack log dataset.

Determine:

1. What distinct warning types are occurring?
2. Which OpenStack services generate them?
3. Which warning pattern is most frequent?
4. Are repeated warnings related to the same resource or operation?
5. Is there evidence in related or nearby log messages that explains
   the recurring warning?
6. What is the most plausible explanation supported by the dataset?

Investigate progressively.

Do not simply retrieve all WARNING records and summarize them.

Start with the warning population, identify important patterns,
quantify them, and drill into the most significant pattern.

Use additional tool calls to verify important hypotheses before
reaching a conclusion.

Clearly distinguish observations from hypotheses and conclusions

In [43]:
question = "What are the top recurring issues in these OpenStack logs, including their services, resources, frequencies, and related events?"
result = run_rlm(
    question,
    max_steps=10
)


═══════════════════════════════════════════════════════════════════════════
  RLM INVESTIGATION STARTED
═══════════════════════════════════════════════════════════════════════════

Question:
What are the top recurring issues in these OpenStack logs, including their services, resources, frequencies, and related events?

Maximum steps: 10

───────────────────────────────────────────────────────────────────────────
  RLM STEP 1/10
───────────────────────────────────────────────────────────────────────────
Model response time: 5.63s

💬 MODEL
I'll investigate the OpenStack logs systematically, starting with a broad structural overview.

🔧 2 tool call(s) requested

🔧 TOOL CALL
   Name      : dataset_summary
   Arguments : {}
   Time      : 0.001s
   Result size: 484 chars

   PYTHON RESULT
   • Lines        : 2000
   • Parsed       : 2000
   • Parse errors : 0
   • Levels       : {'INFO': 1969, 'WARNING': 31}

🔧 TOOL CALL
   Name      : warning_summary
   Arguments : {
  "max_results": 20
}